# Projeto – TP547 Princípios de Simulação de Sistemas de Comunicação

## Análise de Desempenho de Comunicação Cooperativa Full-Duplex em Redes Veiculares Ad-Hoc

**Referência:** S. B. Mafra et al., "Performance Analysis of Full-Duplex Cooperative Communication in Vehicular Ad-Hoc Networks", *IFAC-PapersOnLine*, 2016.

---

### Objetivo

Reproduzir a **Figura 3** do artigo: probabilidade de outage em função da potência de transmissão P (dB), para o **Caso 1** (canal sub-Rayleigh fonte–destino, Rayleigh nos demais), comparando os três esquemas cooperativos:

- **VDH** (*Vehicular Decode-and-Hold*): relé full-duplex; enlace direto visto como interferência no destino.
- **VHD** (*Vehicular Half-Duplex Joint Decoding*): relé half-duplex com protocolo IDF; decodificação conjunta no destino.
- **VJD** (*Vehicular Joint Decoding*): relé full-duplex com decodificação conjunta no destino.
- **Direto**: enlace fonte–destino sem relé (referência).

### Modelo do Sistema

```
          h_sd
Fonte ──────────────────────────────► Destino
  │                                      ▲
  │ h_sr(l)       h_r(l)r(l) (self)      │
  └──────► Relé r(l) ──────────────────┘
                     h_r(l)d
```

Cada canal h_ij segue a **distribuição Nakagami-m**. Os parâmetros do **Caso 1** são:

| Canal | Parâmetro m | Potência média Ω |
|-------|------------|------------------|
| h_sd  | 0,5 (sub-Rayleigh) | d_sd^{−α} = 1 |
| h_sr  | 1 (Rayleigh) | d_sr^{−α} = 16 |
| h_rr  | 1 (auto-interferência) | λ_rr = 10^{−4} |
| h_rd  | 1 (Rayleigh) | d_rd^{−α} = 16 |

**Parâmetros do cenário:** d_sd = 1, d_sr = d_rd = 0,5, α = 4, λ_rr = 10^{−4}, R = 3 bpcu, N_0 = 1, N_relés = 1, P_s = P_r = P.

In [ ]:
!pip install numpy
!pip install matplotlib
!pip install scipy

### Importações

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import nakagami

### Parâmetros do Sistema

In [ ]:
N = 100000       # número de amostras Monte Carlo
N0 = 1           # densidade espectral de potência do ruído
R = 3            # taxa de transmissão (bpcu)
alpha = 4        # expoente de perda de percurso (path loss)

# Distâncias
d_sd = 1.0       # distância fonte-destino
d_sr = 0.5       # distância fonte-relé
d_rd = 0.5       # distância relé-destino

# Potência média dos canais (path loss)
Omega_sd = d_sd**(-alpha)   # = 1
Omega_sr = d_sr**(-alpha)   # = 16
Omega_rd = d_rd**(-alpha)   # = 16
Omega_rr = 1e-4             # auto-interferência no relé (λ_rr)

# Parâmetros Nakagami-m – Caso 1
m_sd = 0.5   # sub-Rayleigh no enlace fonte-destino
m_sr = 1.0   # Rayleigh
m_rr = 1.0   # Rayleigh (auto-interferência)
m_rd = 1.0   # Rayleigh

# Faixa de potência de transmissão
Pdb = np.arange(-5, 21, 1)
Plin = 10**(Pdb / 10)

### Simulação Monte Carlo

Para cada valor de P, geramos N realizações independentes dos canais Nakagami-m e calculamos a probabilidade de outage de cada esquema.

**Informações mútuas usadas (conforme o artigo):**

- Equação (5) – VJD, enlace s→r:
  $$I_{\text{sr}}^{\text{VJD}} = \log_2\!\left(1 + \frac{|h_{sr}|^2 P_s}{|h_{rr}|^2 P_r + N_0}\right)$$

- Equação (6) – VJD, enlace s→d (direto):
  $$I_{\text{sd}}^{\text{VJD}} = \log_2\!\left(1 + \frac{|h_{sd}|^2 P_s}{N_0}\right)$$

- Equação (7) – VJD, enlace r→d com decodificação conjunta:
  $$I_{\text{rd}}^{\text{VJD}} = \log_2\!\left(1 + \frac{|h_{sd}|^2 P_s + |h_{rd}|^2 P_r}{N_0}\right)$$

- Equações (13-14) – VHD (half-duplex, fator 1/2):
  $$I_{sk}^{\text{VHD}} = \frac{1}{2}\log_2\!\left(1 + \frac{|h_{sk}|^2 P_s}{N_0}\right), \quad
  I_{\text{rd}}^{\text{VHD}} = \frac{1}{2}\log_2\!\left(1 + \frac{|h_{sd}|^2 P_s + |h_{rd}|^2 P_r}{N_0}\right)$$

In [ ]:
OP_direct = []
OP_VJD = []
OP_VDH = []
OP_VHD = []

for plin in Plin:
    # Geração dos ganhos de canal |h_ij|^2 com distribuição Nakagami-m
    # Baseado em: outnak.ipynb (codigos_aula13)
    g_sd = (np.sqrt(Omega_sd) * nakagami.rvs(m_sd, size=N))**2
    g_sr = (np.sqrt(Omega_sr) * nakagami.rvs(m_sr, size=N))**2
    g_rd = (np.sqrt(Omega_rd) * nakagami.rvs(m_rd, size=N))**2
    g_rr = (np.sqrt(Omega_rr) * nakagami.rvs(m_rr, size=N))**2

    # --- Enlace Direto (referência) ---
    I_direct = np.log2(1 + g_sd * plin / N0)
    OP_direct.append(np.sum(I_direct < R) / N)

    # --- VJD: relé full-duplex, decodificação conjunta no destino ---
    # eq. (5): informação mútua no relé (com auto-interferência)
    I_sr_VJD = np.log2(1 + g_sr * plin / (g_rr * plin + N0))
    # eq. (6): informação mútua do enlace direto no destino
    I_sd_VJD = np.log2(1 + g_sd * plin / N0)
    # eq. (7): decodificação conjunta no destino (relé + fonte)
    I_rd_VJD = np.log2(1 + (g_sd * plin + g_rd * plin) / N0)
    # Outage: relé decodifica e destino falha na dec. conjunta, OU relé falha e direto falha
    relay_ok_VJD = I_sr_VJD >= R
    O_VJD = np.mean((relay_ok_VJD & (I_rd_VJD < R)) | (~relay_ok_VJD & (I_sd_VJD < R)))
    OP_VJD.append(O_VJD)

    # --- VDH: relé full-duplex, enlace direto como interferência no destino ---
    # Relé recebe com auto-interferência
    I_sr_VDH = np.log2(1 + g_sr * plin / (g_rr * plin + N0))
    # Destino recebe o relé com o enlace direto como interferência
    I_rd_VDH = np.log2(1 + g_rd * plin / (g_sd * plin + N0))
    # eq. (12): outage se o enlace s-r OU o enlace r-d falhar
    O_VDH = np.mean((I_sr_VDH < R) | (I_rd_VDH < R))
    OP_VDH.append(O_VDH)

    # --- VHD: relé half-duplex, protocolo IDF, decodificação conjunta ---
    # eq. (13): fator 1/2 pelo uso de dois slots de tempo
    I_sr_VHD = 0.5 * np.log2(1 + g_sr * plin / N0)
    I_sd_VHD = 0.5 * np.log2(1 + g_sd * plin / N0)
    # eq. (14): decodificação conjunta no destino
    I_rd_VHD = 0.5 * np.log2(1 + (g_sd * plin + g_rd * plin) / N0)
    # Protocolo IDF: relé só retransmite se decodificou com sucesso
    relay_ok_VHD = I_sr_VHD >= R
    O_VHD = np.mean((relay_ok_VHD & (I_rd_VHD < R)) | (~relay_ok_VHD & (I_sd_VHD < R)))
    OP_VHD.append(O_VHD)

### Figura 3 – Probabilidade de Outage × Potência de Transmissão P

In [ ]:
plt.figure(figsize=(8, 6))
plt.semilogy(Pdb, OP_direct, 'k-',  label='Direto')
plt.semilogy(Pdb, OP_VDH,   'r--', label='VDH')
plt.semilogy(Pdb, OP_VHD,   'b-.', label='VHD')
plt.semilogy(Pdb, OP_VJD,   'g-o', markersize=4, label='VJD')
plt.grid(True)
plt.axis([-5, 20, 1e-3, 1])
plt.xlabel('P (dB)')
plt.ylabel('Probabilidade de Outage')
plt.title('Figura 3 – Probabilidade de outage (Caso 1: $m_{sd}=0{,}5$, demais $m=1$)')
plt.legend()
plt.tight_layout()
plt.show()

### Discussão dos Resultados

Os resultados da simulação reproduzem as tendências da Figura 3 do artigo:

1. **VJD** apresenta o melhor desempenho em toda a faixa de P, pois combina a operação full-duplex do relé com a decodificação conjunta dos sinais da fonte e do relé no destino. A auto-interferência no relé é compensada pelo ganho de diversidade da decodificação conjunta.

2. **VHD** tem desempenho inferior ao VJD, principalmente para valores baixos de P, porque o fator 1/2 (dois slots de tempo) penaliza a informação mútua. Para valores altos de P, o VHD pode superar o VDH, pois não sofre do piso de probabilidade causado pela auto-interferência full-duplex e pela interferência do enlace direto.

3. **VDH** apresenta um piso de probabilidade de outage para potências elevadas, causado pela auto-interferência no relé e pela interferência do enlace direto no destino (canal sub-Rayleigh, Caso 1).

4. **Enlace Direto** tem o pior desempenho no Caso 1, pois o canal fonte–destino é sub-Rayleigh ($m_{sd}=0{,}5$), representando condições severas de propagação (obstáculos entre fonte e destino).

**Parâmetros:** $d_{sr}=d_{rd}=0{,}5$, $d_{sd}=1$, $\lambda_{rr}=10^{-4}$, $R=3$ bpcu, $\alpha=4$, $N=100\,000$ amostras.